In [ ]:
import pandas as pd

from sklearn.utils import Bunch

from efaar_benchmarking.constants import COMPOUND_CONCENTRATIONS
from efaar_benchmarking.efaar import pca_centerscale_on_controls
from efaar_benchmarking.benchmarking import known_relationship_benchmark
from efaar_benchmarking.benchmarking import compound_gene_benchmark, BenchmarkConfig

In [ ]:
# be sure to download these files and place them in the data directory, using:
from huggingface_hub import hf_hub_download

file_path_metadata = hf_hub_download("recursionpharma/rxrx3-core", filename="metadata_rxrx3_core.csv",repo_type="dataset",local_dir='./data')
file_path_embs = hf_hub_download("recursionpharma/rxrx3-core", filename="OpenPhenom_rxrx3_core_embeddings.parquet",repo_type="dataset",local_dir='./data')

print(file_path_metadata)
print(file_path_embs)

In [ ]:
rxrx3_metadata = pd.read_csv("data/metadata_rxrx3_core.csv")
openphenom_embeddings = pd.read_parquet("data/OpenPhenom_rxrx3_core_embeddings.parquet")

In [4]:
rxrx3_metadata["perturbation"] = rxrx3_metadata["treatment"].apply(lambda x: x.split("_")[0] if "_control" not in x else x)

In [ ]:
display(openphenom_embeddings)
display(rxrx3_metadata)

In [6]:
results_output_file = "data/RESULT_openphenom__pcacs.pickle"

In [7]:
embeddings_mrged = rxrx3_metadata.merge(
    openphenom_embeddings.groupby("well_id").mean(), 
    left_on="well_id", 
    right_index=True,
)

In [8]:
feature_columns = [c for c in embeddings_mrged.columns if c.startswith("feature_")]
metadata_columns = [c for c in embeddings_mrged.columns if not c.startswith("feature_")]

pert_colname = "perturbation"
experiment_colname = "experiment_name"
control_key = "EMPTY_control"

In [ ]:
print("fitting aligner...")
X = embeddings_mrged[feature_columns].astype(float).values
embeddings_pcacs = pca_centerscale_on_controls(
    X, embeddings_mrged[metadata_columns], pert_col=pert_colname, batch_col=experiment_colname, control_key=control_key
)

assert embeddings_mrged[metadata_columns].shape[0] == embeddings_pcacs.shape[0]

new_metadata = embeddings_mrged[metadata_columns].copy().reset_index()
new_features = pd.DataFrame(embeddings_pcacs, columns=[f"feature_{i}" for i in range(embeddings_pcacs.shape[1])])
aligned_embeddings = pd.concat([new_metadata, new_features], axis=1)

In [10]:
assert aligned_embeddings.feature_0.isna().sum() == 0

# remove controls from henceforth analysis
merged = aligned_embeddings[
    ~(
        (aligned_embeddings["perturbation_type"] == "COMPOUND")
        & (aligned_embeddings[pert_colname].str.contains("_control"))
    )
]

# aggregate to perturbation-level
agg_func = {col: "mean" for col in merged.columns if col.startswith("feature_")}
map_data = (
    merged.groupby(["perturbation_type", pert_colname, "concentration"], dropna=False)
    .agg(agg_func)
    .reset_index()
)
map_data = map_data[map_data.concentration.isin(COMPOUND_CONCENTRATIONS) | map_data.concentration.isna()]
features_cols = [col for col in map_data.columns if col.startswith("feature_")]
metadata_cols = [col for col in map_data.columns if col not in features_cols]

assert map_data.feature_0.isna().sum() == 0

In [ ]:
map_data_gene_only =  map_data.query("perturbation_type == 'CRISPR'")  # only use CRISPR perturbations for gene-gene benchmarks

pert_signal_pval_cutoff = 0.05
recall_thr_pairs = [(0.05, 0.95)]

print("Computing recall...")
bmdb_metrics = known_relationship_benchmark(
    Bunch(metadata=map_data_gene_only[metadata_cols], features=map_data_gene_only[features_cols]),
    recall_thr_pairs=recall_thr_pairs,
    pert_col=pert_colname,
    log_stats=True,
)
print("Recall Results", bmdb_metrics[list(bmdb_metrics.columns)[::-1]])

In [ ]:
all_compound_results = []
for seed in range(100):
    compound_results = compound_gene_benchmark(
        Bunch(metadata=map_data[metadata_cols], features=map_data[features_cols]),
        check_random=False,
        config=BenchmarkConfig(random_seed=seed, quantiles=[0.5, 0.75, 0.95]),
    )
    compound_results["seed"] = seed
    all_compound_results.append(compound_results)
compound_results = pd.concat(all_compound_results)

In [ ]:
compound_results

In [ ]:
median_results = compound_results.groupby('concentration').agg(
    avg_precision_mean=('ap_quantile_0.5', 'mean'),
    avg_precision_std=('ap_quantile_0.5', 'std'),
    baseline_mean=('ap_quantile_0.5_baseline', 'mean'),
    baseline_std=('ap_quantile_0.5_baseline', 'std')
).reset_index()
median_results["concentration"] = median_results["concentration"].astype(str)

upper_quantile_results = compound_results.groupby('concentration').agg(
    avg_precision_mean=('ap_quantile_0.75', 'mean'),
    avg_precision_std=('ap_quantile_0.75', 'std'),
    baseline_mean=('ap_quantile_0.75_baseline', 'mean'),
    baseline_std=('ap_quantile_0.75_baseline', 'std')
).reset_index()
upper_quantile_results["concentration"] = upper_quantile_results["concentration"].astype(str)

mean_results = compound_results.groupby('concentration').agg(
    avg_precision_mean=('average_precision', 'mean'),
    avg_precision_std=('average_precision', 'std'),
    baseline_mean=('average_precision_baseline', 'mean'),
    baseline_std=('average_precision_baseline', 'std')
).reset_index()
mean_results["concentration"] = mean_results["concentration"].astype(str)

mean_results_auc = compound_results.groupby('concentration').agg(
    auc_roc_mean=('auc_roc', 'mean'),
    auc_roc_std=('auc_roc', 'std'),
    baseline_auc_mean=('auc_roc_baseline', 'mean'),
    baseline_auc_std=('auc_roc_baseline', 'std')
).reset_index()
mean_results_auc["concentration"] = mean_results_auc["concentration"].astype(str)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Define the metrics and their corresponding labels
metrics = [
    {
        "mean_results": median_results,
        "y_mean": "avg_precision_mean",
        "y_std": "avg_precision_std",
        "baseline_mean": "baseline_mean",
        "baseline_std": "baseline_std",
        "title": "Median Average Precision vs Baseline Precision",
        "yaxis_title": "Median Avg. Precision"
    },
    {
        "mean_results": upper_quantile_results,
        "y_mean": "avg_precision_mean",
        "y_std": "avg_precision_std",
        "baseline_mean": "baseline_mean",
        "baseline_std": "baseline_std",
        "title": "Upper quantile of Average Precision vs Baseline Precision",
        "yaxis_title": "Upper quantile Avg. Precision"
    },
    {
        "mean_results": mean_results,
        "y_mean": "avg_precision_mean",
        "y_std": "avg_precision_std",
        "baseline_mean": "baseline_mean",
        "baseline_std": "baseline_std",
        "title": "Mean Average Precision vs Baseline",
        "yaxis_title": "Mean Avg. Precision"
    },
    {
        "mean_results": mean_results_auc,
        "y_mean": "auc_roc_mean",
        "y_std": "auc_roc_std",
        "baseline_mean": "baseline_auc_mean",
        "baseline_std": "baseline_auc_std",
        "title": "Mean AUC ROC vs Baseline AUC ROC",
        "yaxis_title": "Mean AUC ROC"
    }
]

# Loop through the metrics and create the figures
for metric in metrics:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=metric["mean_results"]['concentration'],
        y=metric["mean_results"][metric["y_mean"]],
        error_y=dict(type='data', array=metric["mean_results"][metric["y_std"]]),
        mode='lines+markers',
        name="OpenPhenom",
    ))

    fig.add_trace(go.Scatter(
        x=metric["mean_results"]['concentration'],
        y=metric["mean_results"][metric["baseline_mean"]],
        error_y=dict(type='data', array=metric["mean_results"][metric["baseline_std"]]),
        mode='lines+markers',
        name="Random Baseline",
    ))

    fig.update_layout(
        title=metric["title"],
        xaxis_title='Concentration',
        yaxis_title=metric["yaxis_title"],
        template='plotly_dark'
    )

    fig.show()